# Thesis Figure Refresh (B/W)

**Source script:** `scripts/refresh_chapter_figures_bw.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ROOT = Path('/Users/omaralneser/Documents/TYP')
DATASET = ROOT / 'outputs/eda_v3_d7_compare/datasets/d7_second_only.csv'
EDA = ROOT / 'outputs/eda_v3_d7_compare/d7_second_only/eda_v3'
FIG = ROOT / 'thesis-paper/figures'

DISEASE_ORDER = [
    'acute pancreatitis',
    'acute flare-up of triaditis/chronic enteropathy',
    'acute gastroenteritis',
    'parvovirus enteritis',
]
DISEASE_LABEL = {
    'acute pancreatitis': 'acute pancreatitis',
    'acute flare-up of triaditis/chronic enteropathy': 'triaditis/CE flare-up',
    'acute gastroenteritis': 'acute gastroenteritis',
    'parvovirus enteritis': 'parvovirus enteritis',
}

plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})


def load_env_numeric() -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_csv(DATASET)
    if 'moon_phase_sin' not in df.columns or 'moon_phase_cos' not in df.columns:
        if 'moon_phase_angle_deg' in df.columns:
            ang = pd.to_numeric(df['moon_phase_angle_deg'], errors='coerce')
            rad = np.deg2rad(ang)
            df['moon_phase_sin'] = np.sin(rad)
            df['moon_phase_cos'] = np.cos(rad)

    kept = pd.read_csv(EDA / 'tables/weather_features_kept_after_correlation.csv')['kept_feature'].tolist()
    X = df[kept].apply(pd.to_numeric, errors='coerce')
    X = X.fillna(X.median(numeric_only=True))
    y = df['group'].astype(str).values

    Xs = StandardScaler().fit_transform(X)
    Xp = PCA(n_components=6, random_state=42).fit_transform(Xs)
    return df, X.to_numpy(), Xp, y


def plot_corr_grouped() -> None:
    corr = pd.read_csv(EDA / 'tables/weather_correlation_matrix_abs.csv', index_col=0)


    main = FIG / 'corr_heatmap_weather_topvariance.pdf'
    appendix_copy = FIG / 'corr_heatmap_weather_topvariance_grouped_appendix.pdf'
    if main.exists():
        shutil.copy2(main, appendix_copy)

    cols = corr.columns.tolist()

    def family_name(feature: str) -> str:
        base = feature.split('__')[0]
        return 'moon_phase' if base in {'moon_phase_sin', 'moon_phase_cos'} else base

    mat = corr.loc[cols, cols].to_numpy()
    fig, ax = plt.subplots(figsize=(13.2, 12.4))
    im = ax.imshow(mat, cmap='Greys', vmin=0, vmax=1, interpolation='nearest', aspect='equal')

    ax.set_xticks(np.arange(len(cols)))
    ax.set_yticks(np.arange(len(cols)))
    ax.set_xticklabels(cols, rotation=90, fontsize=6)
    ax.set_yticklabels(cols, fontsize=6)


    boundaries = []
    prev = family_name(cols[0]) if cols else ''
    for i, c in enumerate(cols[1:], start=1):
        fam = family_name(c)
        if fam != prev:
            boundaries.append(i - 0.5)
        prev = fam
    for b in boundaries:
        ax.axhline(b, color='white', linewidth=0.75)
        ax.axvline(b, color='white', linewidth=0.75)

    ax.set_title('Absolute correlation heatmap (top-variance weather features)')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('|r|')
    fig.tight_layout()
    fig.savefig(main, format='pdf', bbox_inches='tight', dpi=600)
    plt.close(fig)


def plot_pca_scatter(Xp: np.ndarray, y: np.ndarray) -> None:
    markers = ['o', 's', '^', 'D']
    shades = ['#111111', '#4d4d4d', '#8a8a8a', '#c7c7c7']

    fig, ax = plt.subplots(figsize=(8.2, 5.8))
    for disease, mk, color in zip(DISEASE_ORDER, markers, shades):
        m = y == disease
        ax.scatter(
            Xp[m, 0], Xp[m, 1],
            marker=mk,
            c=color,
            edgecolors='black',
            linewidths=0.5,
            s=32,
            alpha=1.0,
            label=DISEASE_LABEL[disease],
        )

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('Environmental-only PCA (PC1 vs PC2) by diagnosis')
    ax.grid(True, color='#d9d9d9', linestyle='--', linewidth=0.6)
    ax.legend(frameon=True, edgecolor='black', facecolor='white', loc='upper right')
    fig.tight_layout()
    fig.savefig(FIG / 'pca_pc1_pc2_env.pdf', format='pdf', bbox_inches='tight')
    plt.close(fig)


def plot_kmeans(Xp: np.ndarray, k: int, out_name: str, title: str) -> None:
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = km.fit_predict(Xp[:, :6])

    marker_list = ['o', 's', '^', 'D', 'v', 'P', 'X', '<', '>', '*']
    shade_list = ['#111111', '#3d3d3d', '#666666', '#8a8a8a', '#b0b0b0', '#d2d2d2', '#4f4f4f', '#767676', '#9d9d9d', '#262626']

    fig, ax = plt.subplots(figsize=(8.2, 5.8))
    for c in sorted(np.unique(labels)):
        m = labels == c
        ax.scatter(
            Xp[m, 0], Xp[m, 1],
            marker=marker_list[c % len(marker_list)],
            c=shade_list[c % len(shade_list)],
            edgecolors='black',
            linewidths=0.5,
            s=32,
            alpha=1.0,
            label=f'C{c}'
        )

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)
    ax.grid(True, color='#d9d9d9', linestyle='--', linewidth=0.6)
    ax.legend(frameon=True, edgecolor='black', facecolor='white', loc='upper right', ncol=2)
    fig.tight_layout()
    fig.savefig(FIG / out_name, format='pdf', bbox_inches='tight')
    plt.close(fig)


def main() -> None:
    df, X, Xp, y = load_env_numeric()
    plot_corr_grouped()
    plot_pca_scatter(Xp, y)

    sil = pd.read_csv(EDA / 'unsupervised/tables/kmeans_silhouette_environmental_only_Dfixed6.csv')
    kbest = int(sil.loc[sil['silhouette_score'].idxmax(), 'k'])
    plot_kmeans(Xp, kbest, 'kmeans_env_kbest_dfixed6.pdf', f'Environmental-only KMeans (Dfixed6, k_best={kbest})')
    plot_kmeans(Xp, 6, 'kmeans_env_k6_dfixed6.pdf', 'Environmental-only KMeans (Dfixed6, k=6)')


if __name__ == '__main__':
    main()
